In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import numba
import molsim


<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Exercise 5: Random Walk on a 2D lattice

Consider the random walk of $ N $ particles on a $M \times M $ lattice. Two particles cannot occupy the same lattice site. On this lattice, periodic boundaries are used. This means that when a particle leaves the lattice, it returns on the opposite side of the lattice.

<div style="max-width:300px;margin-right: auto; margin-right: 0;">


</div>

**Figure:** A random walk of $ N $ particles on a $ M \times M $ lattice. One of the particles is chosen at random (the green one). Next, one of the four (up, down, left, right) directions is chosen. Only if the site is empty, the jump is allowed. In this example, the jump of the green particle to the left would be rejected. Particles leaving the lattice are put back at the other end of the lattice (periodic boundary conditions).

#### Questions:

1. What is the fraction of occupied sites ($ \theta $) of the lattice as a function of $ M $ and $ N $?

2. Make a plot of the diffusion coefficient $ D $ as a function of $\theta $ for $M = 32 $. For low values of $ \theta $, the diffusion coefficient can be approximated by:

   \begin{equation}\tag{5}
   D \approx D_0 (1 - \theta),
   \end{equation}

   where $ D_0 $ is the diffusion coefficient at infinite dilution (one particle on an empty lattice). Derive this equation. Why is this equation not exact?
</div>


In [ ]:
@numba.njit
def randomWalk2D(
    numberOfCycles: int,
    numberOfParticles: int,
    latticeSize: int,
    maxCorrelationTime: int = 500,
    maxOrigins: int = 50,
    originInterval: int = 50,
    fractionOfFrozenParticles: float = 0.0,
):
    """
    Simulate a 2D random walk of multiple particles on a lattice and compute mean squared displacement (MSD).

    This simulation considers a set of particles placed on a 2D lattice of size `latticeSize x latticeSize`.
    At each cycle, a random particle attempts to move to a neighboring lattice site (up, down, left, or right).
    The move is accepted if the target site is not occupied. Occupation is tracked in a lattice array.

    The MSD is sampled by periodically recording "origin" configurations and measuring the displacement
    from these origins after different time intervals.

    Parameters
    ----------
    numberOfCycles : int
        The number of Monte Carlo cycles (attempted moves) to simulate.
    numberOfParticles : int
        The number of particles placed on the lattice.
    latticeSize : int
        The dimension of the square lattice (latticeSize x latticeSize).
    maxCorrelationTime : int, optional
        The maximum time interval for which to compute the MSD. Default is 500.
    maxOrigins : int, optional
        The maximum number of origin configurations stored for MSD calculations. Default is 50.
    originInterval:
        The interval (in cycles) at which origins are recorded for MSD calculations.
    fractionOfFrozenParticles: int, optional
        The fraction of randomly selected particles that are frozen in place.

    Returns
    -------
    msd : ndarray
        An array of shape (maxCorrelationTime, 2) giving the MSD in x and y directions
        at different time intervals.
    """
    # Define possible moves: right,left,up,down
    latticeVectors = np.array([[1, 0], [-1, 0], [0, 1], [0, -1]], dtype=np.int32)
    # Initialize an empty lattice: 0 means empty site, 1 means occupied.
    lattice = np.zeros((latticeSize, latticeSize), dtype=np.int32)

    # Randomly choose distinct initial positions for all particles.
    indices = np.random.choice(latticeSize**2, size=numberOfParticles, replace=False)

    # Select frozen indices
    # start refactor
    numberOfFrozenParticles = None
    frozen_indices = None
    frozen_set = None
    # end refactor

    xPositions = indices % latticeSize
    yPositions = indices // latticeSize

    # positions holds the current wrapped positions of each particle within the lattice.
    positions = np.column_stack((xPositions, yPositions))
    # unwrappedPositions keeps track of net displacement without wrapping.
    unwrappedPositions = positions.copy()

    # Initialize arrays for MSD sampling.
    msd = np.zeros((maxCorrelationTime, 2))
    counts = np.zeros(maxCorrelationTime, dtype=np.int32)
    originPositions = np.zeros((maxOrigins, numberOfParticles, 2))
    originTimes = np.zeros(maxOrigins, dtype=np.int32)
    originIndex = 0

    accepted = 0  # Count the number of accepted moves.

    # Populate the lattice with initial particles.
    for px, py in positions:
        lattice[px, py] = 1

    # Main simulation loop.
    for cycle in range(numberOfCycles):
        # Select a random particle to attempt a move.
        particleIndex = np.random.choice(numberOfParticles)

        # skip if in frozen particles
        # start implementation
        # end implementation

        # Select a random direction to attempt (right/left/up/down).
        # option to change, favouring the x-direction (implementation 3)
        # start refactor
        probabilities = np.array([0.25, 0.25, 0.25, 0.25])
        # end refactor

        # find idx in lattice array
        cumulative_probs = np.cumsum(probabilities)
        r = np.random.random()
        idx = np.searchsorted(cumulative_probs, r)
        dx = latticeVectors[idx]

        xold, yold = positions[particleIndex]

        # Compute proposed new position
        newPosition = positions[particleIndex] + dx

        # start implementation
        # end implementation

        # Wrap it in the lattice
        newPosition = newPosition % latticeSize
        xnew, ynew = newPosition

        # Check if the proposed site is free.
        if lattice[xnew, ynew] == 0:
            # Accept move: update lattice and particle positions.
            accepted += 1
            lattice[xold, yold] = 0
            lattice[xnew, ynew] = 1
            positions[particleIndex] = newPosition
            unwrappedPositions[particleIndex] += dx

        # Start sampling MSD after the system has somewhat equilibrated (25% of runs).
        if cycle > 0.25 * numberOfCycles:
            # Record a new origin configuration periodically.
            if cycle % originInterval == 0:
                originTimes[originIndex] = cycle
                originPositions[originIndex] = unwrappedPositions
                originIndex = (originIndex + 1) % maxOrigins

            # Compute MSD for each origin recorded so far, if time intervals are valid.
            for i in range(min(cycle // maxOrigins, maxOrigins)):
                time_difference = cycle - originTimes[i]
                if time_difference < maxCorrelationTime:
                    counts[time_difference] += 1
                    # Sum the squared displacements from origin positions.
                    msd[time_difference] += np.sum((unwrappedPositions - originPositions[i]) ** 2, axis=0)

    # Average the MSD by dividing by counts where counts are non-zero.
    nonZero = counts > 0
    msd[nonZero] /= counts[nonZero][:, None]

    # Print some diagnostic information.
    print(f"Total accepted: {accepted}\nLattice occupation: {np.sum(lattice)}")

    return msd

In [ ]:
numberOfParticles = 50
latticeSize = 10
numberOfCycles = 1000000
msd = randomWalk2D(numberOfCycles, numberOfParticles, latticeSize)

In [ ]:
# Plot the MSD in x and y directions.
fig, ax = plt.subplots()
ax.plot(msd[:, 0], label="x-direction")
ax.plot(msd[:, 1], label="y-direction")
ax.set_ylabel("MSD")
ax.set_xlabel("Cycle")
plt.legend()
plt.show()

In [ ]:
numberOfDensities = 5
numberOfParticlesList = np.linspace(1, latticeSize**2, numberOfDensities)
occupancy = numberOfParticlesList / latticeSize**2
result_diffusion_x = np.zeros(numberOfDensities)
result_diffusion_y = np.zeros(numberOfDensities)

for index, numberOfParticles in enumerate(numberOfParticlesList):
    msd = randomWalk2D(numberOfCycles, int(numberOfParticles), latticeSize)
    # calculate diffusion constant
    # start implementation
    # end implementation

In [ ]:
fig, ax = plt.subplots()

# Plot the data with custom styles
# start implementation
# end implementation
ax.plot(occupancy, 0.25 * (1.0 - occupancy), label=r"$D_0(1-\theta)$", c="black")

# Add labels, title, and legend
ax.set_xlabel(r"Occupancy $\theta$")
ax.set_ylabel("Diffusion coefficient $D$")
ax.set_title("Diffusion in x and y directions")
ax.legend()

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

3. Modify the program in such a way that the probability to jump in the $ x $ direction is larger than the probability to jump in the $ y $ direction. Explain the results.

4. Modify the program in such a way that periodic boundary conditions are used in one direction only, and make the lattice boundaries hard walls in the other directions, so that particles cannot escape, i.e., moves in these directions that cross the lattice boundaries are rejected. What happens?

5. Modify the program in such a way that a certain fraction of the particles are “frozen.” Investigate the influence of the fraction of frozen particles on the diffusion coefficient.

</div>